## 1) Install packages

In [ ]:
!pip -q install -U \
  requests==2.32.4 \
  trafilatura \
  beautifulsoup4 \
  lxml \
  rank_bm25 \
  sentence-transformers \
  transformers \
  accelerate \
  bitsandbytes \
  faiss-cpu \
  langchain \
  langchain-community \
  langchain-huggingface \
  pandas \
  datasets

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.5/79.5 kB 6.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 132.6/132.6 kB 16.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 107.7/107.7 kB 13.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.2/5.2 MB 96.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 571.3/571.3 kB 53.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 111.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 12.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 77.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 68.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 112.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 459.1/459.1 kB 45.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.9/10.9 MB 134.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52

## 2) Restart runtime once

In [ ]:
import os
os.kill(os.getpid(), 9)

## 3) Imports

In [ ]:
import os
import re
import json
import time
import torch
import requests
import trafilatura
import numpy as np
import pandas as pd

from pathlib import Path
from urllib.parse import urljoin, urlparse, urldefrag

from bs4 import BeautifulSoup
from rank_bm25 import BM25Okapi
from datasets import Dataset
from sentence_transformers import CrossEncoder

from langchain_core.documents import Document
from langchain_community.vectorstores import FAISS
from langchain_huggingface import HuggingFaceEmbeddings

from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    BitsAndBytesConfig,
    pipeline
)

print("Torch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
print("Requests version:", requests.__version__)

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("Enable GPU in Colab: Runtime > Change runtime type > GPU")

Torch version: 2.10.0+cu128
CUDA available: True
Requests version: 2.32.4
GPU: Tesla T4


## 4) Project config

In [ ]:
BASE_DIR = Path("/content/crocoit_rag_final")
DATA_DIR = BASE_DIR / "data"
FAISS_DIR = BASE_DIR / "faiss_index"

DATA_DIR.mkdir(parents=True, exist_ok=True)
FAISS_DIR.mkdir(parents=True, exist_ok=True)

SEED_URLS = [
    "https://crocoit.com/",
    "https://crocoit.com/solutions/",
    "https://crocoit.com/about-us/",
    "https://crocoit.com/our-products/",
    "https://crocoit.com/articles/",
    "https://crocoit.com/case-studies/",
]

ALLOWED_DOMAINS = {
    "crocoit.com",
    "www.crocoit.com",
}

BLOCKLIST_KEYWORDS = [
    "/wp-admin",
    "/feed",
    "/tag/",
    "/author/",
    "/category/",
    "/comments",
    "/privacy-policy",
    "/terms",
    "/login",
    "/register",
    "/signup",
    "/cart",
    "/checkout",
    "mailto:",
    "tel:",
    "#",
]

MAX_PAGES = 80
REQUEST_TIMEOUT = 20
SLEEP_BETWEEN_REQUESTS = 0.6

EMBED_MODEL_NAME = "BAAI/bge-m3"
RERANKER_MODEL_NAME = "BAAI/bge-reranker-v2-m3"
LLM_MODEL_NAME = "Qwen/Qwen2.5-3B-Instruct"

print("Base directory:", BASE_DIR)
for u in SEED_URLS:
    print("-", u)
print("Reranker:", RERANKER_MODEL_NAME)

Base directory: /content/crocoit_rag_final
- https://crocoit.com/
- https://crocoit.com/solutions/
- https://crocoit.com/about-us/
- https://crocoit.com/our-products/
- https://crocoit.com/articles/
- https://crocoit.com/case-studies/
Reranker: BAAI/bge-reranker-v2-m3


## 5) Crawling and cleaning helpers

In [ ]:
session = requests.Session()
session.headers.update({
    "User-Agent": "Mozilla/5.0 (compatible; CrocoIT-RAG-Bot/1.0; +https://colab.research.google.com/)"
})

def normalize_url(url: str) -> str:
    url, _ = urldefrag(url)
    parsed = urlparse(url)
    scheme = parsed.scheme or "https"
    netloc = parsed.netloc.lower()
    path = parsed.path or "/"
    if path != "/" and path.endswith("/"):
        path = path[:-1]
    return f"{scheme}://{netloc}{path}"

def path_allowed(path: str) -> bool:
    if path in {"", "/"}:
        return True
    for prefix in ["/solutions", "/about-us", "/our-products", "/articles", "/case-studies"]:
        if path == prefix or path.startswith(prefix + "/"):
            return True
    return False

def is_allowed_url(url: str) -> bool:
    try:
        parsed = urlparse(url)

        if parsed.scheme not in {"http", "https"}:
            return False

        if parsed.netloc.lower() not in ALLOWED_DOMAINS:
            return False

        lowered = url.lower()

        if any(k in lowered for k in BLOCKLIST_KEYWORDS):
            return False

        if re.search(r"\.(jpg|jpeg|png|gif|svg|webp|css|js|xml|zip|rar|mp4|mp3|pdf)$", lowered):
            return False

        if not path_allowed(parsed.path):
            return False

        return True
    except Exception:
        return False

def classify_page(url: str) -> str:
    path = urlparse(url).path.lower()

    if path in {"", "/"}:
        return "home"
    if path.startswith("/solutions"):
        return "service"
    if path.startswith("/about-us"):
        return "about"
    if path.startswith("/our-products"):
        return "product"
    if path.startswith("/articles"):
        return "blog"
    if path.startswith("/case-studies"):
        return "case_study"
    return "general"

def fetch_html(url: str) -> str | None:
    try:
        response = session.get(url, timeout=REQUEST_TIMEOUT)
        content_type = response.headers.get("Content-Type", "")
        if response.status_code == 200 and "text/html" in content_type:
            return response.text
        return None
    except Exception:
        return None

def extract_title(html: str, fallback: str = "") -> str:
    soup = BeautifulSoup(html, "html.parser")
    if soup.title and soup.title.get_text(strip=True):
        return soup.title.get_text(" ", strip=True)
    h1 = soup.find("h1")
    if h1:
        return h1.get_text(" ", strip=True)
    return fallback

def extract_links(base_url: str, html: str) -> list[str]:
    soup = BeautifulSoup(html, "html.parser")
    links = set()
    for a in soup.find_all("a", href=True):
        href = a["href"].strip()
        full = normalize_url(urljoin(base_url, href))
        if is_allowed_url(full):
            links.add(full)
    return sorted(links)

def extract_headings(html: str):
    soup = BeautifulSoup(html, "html.parser")
    headings = []
    for tag in soup.find_all(["h1", "h2", "h3"]):
        text = tag.get_text(" ", strip=True)
        if text and len(text) > 2:
            headings.append(text)
    return headings

def extract_main_text(html: str) -> str:
    extracted = trafilatura.extract(
        html,
        include_comments=False,
        include_tables=True,
        include_links=False,
        include_images=False,
        favor_precision=True,
        deduplicate=True
    )
    return extracted.strip() if extracted else ""

def is_boilerplate_line(line: str) -> bool:
    low = line.lower().strip()

    bad_patterns = [
        r"^\s*$",
        r"^menu$",
        r"^search$",
        r"^close$",
        r"^skip to content$",
        r"^read more$",
        r"^learn more$",
        r"^share$",
        r"^follow us$",
        r"^contact us$",
        r"^get started$",
        r"^book a demo$",
        r"^request a quote$",
        r"^all rights reserved$",
        r"^copyright.*$",
        r"^privacy policy$",
        r"^terms.*$",
        r"^cookie.*$",
        r"^subscribe.*$",
        r"^newsletter.*$",
        r"^home$",
        r"^about us$",
        r"^our services$",
        r"^our products$",
        r"^articles$",
        r"^case studies$",
    ]

    if any(re.match(p, low) for p in bad_patterns):
        return True

    if len(low) <= 1:
        return True

    # mostly contact-only lines
    if re.match(r"^(phone|email|address|location)\s*[:\-].*$", low):
        return True

    # pure email / phone / URL lines
    if re.match(r"^[\w\.-]+@[\w\.-]+\.\w+$", low):
        return True
    if re.match(r"^\+?\d[\d\-\s\(\)]{6,}$", low):
        return True
    if re.match(r"^(https?://|www\.).*$", low):
        return True

    return False

def remove_boilerplate_lines(text: str) -> str:
    lines = [line.strip() for line in text.splitlines()]
    cleaned = []

    for line in lines:
        if is_boilerplate_line(line):
            continue
        cleaned.append(line)

    # remove repeated lines
    deduped = []
    seen = set()
    for line in cleaned:
        key = line.lower()
        if key in seen:
            continue
        seen.add(key)
        deduped.append(line)

    return "\n".join(deduped)

def clean_text(text: str) -> str:
    text = remove_boilerplate_lines(text)
    text = re.sub(r"\n{3,}", "\n\n", text)
    text = re.sub(r"[ \t]{2,}", " ", text)
    return text.strip()

def detect_language_hint(text: str) -> str:
    return "en"

## 6) Crawl CrocoIT pages

In [ ]:
to_visit = [normalize_url(u) for u in SEED_URLS if is_allowed_url(u)]
visited = set()
scraped_docs = []

while to_visit and len(visited) < MAX_PAGES:
    url = to_visit.pop(0)

    if url in visited:
        continue

    print(f"Scraping: {url}")
    visited.add(url)

    html = fetch_html(url)
    if not html:
        print("  -> skipped (no HTML)")
        continue

    raw_text = extract_main_text(html)
    text = clean_text(raw_text)
    title = extract_title(html, fallback=url)
    headings = extract_headings(html)

    if len(text) < 120:
        print("  -> skipped (too little content)")
        continue

    doc = {
        "url": url,
        "title": title,
        "text": text,
        "headings": headings,
        "domain": urlparse(url).netloc.lower(),
        "page_type": classify_page(url),
        "language_hint": detect_language_hint(text),
    }
    scraped_docs.append(doc)

    for link in extract_links(url, html):
        if link not in visited and link not in to_visit:
            to_visit.append(link)

    time.sleep(SLEEP_BETWEEN_REQUESTS)

print("\nVisited:", len(visited))
print("Scraped docs kept:", len(scraped_docs))

with open(DATA_DIR / "scraped_pages.json", "w", encoding="utf-8") as f:
    json.dump(scraped_docs, f, ensure_ascii=False, indent=2)

print("Saved:", DATA_DIR / "scraped_pages.json")

Scraping: https://crocoit.com/
Scraping: https://crocoit.com/solutions
Scraping: https://crocoit.com/about-us
Scraping: https://crocoit.com/our-products
Scraping: https://crocoit.com/articles
  -> skipped (too little content)
Scraping: https://crocoit.com/case-studies
  -> skipped (too little content)

Visited: 6
Scraped docs kept: 4
Saved: /content/crocoit_rag_final/data/scraped_pages.json


## 7) Inspect scraped content

In [ ]:
for i, doc in enumerate(scraped_docs[:10], start=1):
    print(f"\n[{i}] {doc['title']}")
    print("URL:", doc["url"])
    print("Page type:", doc["page_type"])
    print("Headings:", doc["headings"][:5])
    print(doc["text"][:700], "...\n")


[1] CrocoIT - CrocoIT
URL: https://crocoit.com/
Page type: home
Headings: ['Unlock Insights and Secure Loyalty', 'Business Coach', 'Web Design', 'Web Design', 'Typography']
Experience the future of digital transformation with ARTRIXX—a comprehensive AI suite built for seamless cross-platform performance on both iOS and Android. Featuring intelligent chatbots, dynamic AI image search, immersive virtual makeup try-on, and robust retail system analysis, ARTRIXX empowers your business to optimize customer interactions and streamline operations. Discover innovative tools that elevate your online presence and drive growth. ...


[2] Solutions – CrocoIT
URL: https://crocoit.com/solutions
Page type: service
Headings: ['Start Your Business', 'Business Coaching', 'Phone Consultation', 'Marketing Planning', 'Marketplace Management']
Transform your business with our expert consultation and audit services. Our team of seasoned consultants provides in-depth digital transformation consultation, IT a

## 8) Section-aware chunking

In [ ]:
def split_into_sections(title: str, headings: list, text: str):
    lines = [x.strip() for x in text.splitlines() if x.strip()]
    heading_set = set([h.strip() for h in headings if h.strip()])

    sections = []
    current_heading = title
    current_lines = []

    for line in lines:
        if line in heading_set and len(current_lines) > 0:
            sections.append({
                "section_heading": current_heading,
                "section_text": "\n".join(current_lines).strip()
            })
            current_heading = line
            current_lines = []
        else:
            current_lines.append(line)

    if current_lines:
        sections.append({
            "section_heading": current_heading,
            "section_text": "\n".join(current_lines).strip()
        })

    return sections

def chunk_text_with_overlap(text, chunk_size=400, overlap=80):
    chunks = []
    start = 0
    n = len(text)

    while start < n:
        end = min(start + chunk_size, n)
        chunk = text[start:end].strip()
        if chunk:
            chunks.append(chunk)
        if end == n:
            break
        start = max(end - overlap, start + 1)

    return chunks

chunk_docs = []
chunk_id = 0

for doc in scraped_docs:
    sections = split_into_sections(
        title=doc["title"],
        headings=doc["headings"],
        text=doc["text"]
    )

    for sec_idx, sec in enumerate(sections):
        section_chunks = chunk_text_with_overlap(
            sec["section_text"],
            chunk_size=400,
            overlap=80
        )

        for local_idx, chunk in enumerate(section_chunks):
            content = (
                f"Title: {doc['title']}\n"
                f"URL: {doc['url']}\n"
                f"Page Type: {doc['page_type']}\n"
                f"Section Heading: {sec['section_heading']}\n\n"
                f"{chunk}"
            )

            metadata = {
                "chunk_id": chunk_id,
                "source": doc["url"],
                "title": doc["title"],
                "page_type": doc["page_type"],
                "domain": doc["domain"],
                "language_hint": doc["language_hint"],
                "section_heading": sec["section_heading"],
                "section_index": sec_idx,
                "local_chunk_index": local_idx,
            }

            chunk_docs.append(Document(page_content=content, metadata=metadata))
            chunk_id += 1

print("Total chunks:", len(chunk_docs))
print("\nSample chunk:\n")
print(chunk_docs[0].page_content[:1000])
print("\nMetadata:\n", chunk_docs[0].metadata)

Total chunks: 12

Sample chunk:

Title: CrocoIT - CrocoIT
URL: https://crocoit.com/
Page Type: home
Section Heading: CrocoIT - CrocoIT

Experience the future of digital transformation with ARTRIXX—a comprehensive AI suite built for seamless cross-platform performance on both iOS and Android. Featuring intelligent chatbots, dynamic AI image search, immersive virtual makeup try-on, and robust retail system analysis, ARTRIXX empowers your business to optimize customer interactions and streamline operations. Discover innovative tools

Metadata:
 {'chunk_id': 0, 'source': 'https://crocoit.com/', 'title': 'CrocoIT - CrocoIT', 'page_type': 'home', 'domain': 'crocoit.com', 'language_hint': 'en', 'section_heading': 'CrocoIT - CrocoIT', 'section_index': 0, 'local_chunk_index': 0}


## 9) Build embeddings and FAISS

In [ ]:
device = "cuda" if torch.cuda.is_available() else "cpu"

embeddings = HuggingFaceEmbeddings(
    model_name=EMBED_MODEL_NAME,
    model_kwargs={"device": device},
    encode_kwargs={"normalize_embeddings": True}
)

vectorstore = FAISS.from_documents(
    documents=chunk_docs,
    embedding=embeddings
)

vectorstore.save_local(str(FAISS_DIR))
print("FAISS index saved to:", FAISS_DIR)

Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

FAISS index saved to: /content/crocoit_rag_final/faiss_index


## 10) BM25 index

In [ ]:
def simple_tokenize(text: str):
    text = text.lower()
    text = re.sub(r"[^\w\u0600-\u06FF\s]", " ", text)
    return text.split()

bm25_corpus = [simple_tokenize(doc.page_content) for doc in chunk_docs]
bm25 = BM25Okapi(bm25_corpus)

print("BM25 index ready.")

BM25 index ready.


## 11) Query expansion

In [ ]:
QUERY_HINTS = {
    "services": ["solutions", "consultation", "audit", "erp", "marketplace", "marketing", "services"],
    "products": ["our products", "2loyal", "artrixx", "platform", "software", "products"],
    "about": ["about us", "mission", "vision", "company", "who we are", "crocoit"],
    "case studies": ["portfolio", "case studies", "projects", "success stories"],
    "blogs": ["articles", "blog", "insights", "news"],
    "what is 2loyal": ["2loyal", "loyalty", "rewards", "customers", "retention"],
    "what is artrixx": ["artrixx", "product", "platform", "software"],
}

KNOWN_ENTITIES = [
    "crocoit",
    "2loyal",
    "artrixx",
    "erp",
]

def extract_query_entities(query: str):
    q_low = query.lower()
    found = []

    for ent in KNOWN_ENTITIES:
        if ent in q_low:
            found.append(ent)

    # capture quoted phrases if any
    quoted = re.findall(r'"([^"]+)"', query)
    for q in quoted:
        q = q.strip().lower()
        if q and q not in found:
            found.append(q)

    # lightweight title-case phrase capture
    words = re.findall(r"\b[A-Za-z][A-Za-z0-9\-]+\b", query)
    for w in words:
        wl = w.lower()
        if wl in {"what", "does", "is", "the", "a", "an", "of", "about", "say", "mention"}:
            continue
        if wl not in found and (wl in KNOWN_ENTITIES or any(ch.isdigit() for ch in wl)):
            found.append(wl)

    return list(dict.fromkeys(found))

def classify_query_type(query: str):
    q = query.lower().strip()

    if q.startswith("what is ") or q.startswith("define ") or q.startswith("tell me about "):
        return "definition"

    if any(x in q for x in ["what services", "which services", "list services", "services does"]):
        return "service_listing"

    if any(x in q for x in ["what products", "which products", "list products", "products does"]):
        return "product_listing"

    if any(x in q for x in ["about us", "mission", "vision", "who is crocoit", "who are crocoit", "company", "background"]):
        return "about_company"

    if any(x in q for x in ["case study", "case studies", "portfolio", "project", "projects", "success story"]):
        return "case_study"

    if any(x in q for x in ["blog", "blogs", "article", "articles", "news", "insight", "insights"]):
        return "blog"

    return "general"

def expand_query(query: str):
    q_low = query.lower()
    expanded = [query]
    q_type = classify_query_type(query)
    entities = extract_query_entities(query)

    for k, vals in QUERY_HINTS.items():
        if k in q_low:
            expanded.extend(vals)

    if q_type == "definition":
        expanded.extend(["definition", "overview", "what is", "product", "platform"])
    elif q_type == "service_listing":
        expanded.extend(["services", "solutions", "offerings", "what we offer"])
    elif q_type == "product_listing":
        expanded.extend(["products", "platforms", "software", "our products"])
    elif q_type == "about_company":
        expanded.extend(["about us", "mission", "vision", "company"])
    elif q_type == "case_study":
        expanded.extend(["case studies", "portfolio", "projects"])
    elif q_type == "blog":
        expanded.extend(["articles", "blogs", "insights"])

    expanded.extend(entities)

    if "crocoit" not in q_low:
        expanded.append("CrocoIT")

    expanded = list(dict.fromkeys([x.strip() for x in expanded if x.strip()]))
    return expanded

## 12) Page-type preference detection

In [ ]:
def detect_page_type_preferences(query: str):
    q_low = query.lower()
    q_type = classify_query_type(query)

    prefs = {
        "home": 0.0,
        "service": 0.0,
        "about": 0.0,
        "product": 0.0,
        "blog": 0.0,
        "case_study": 0.0,
        "general": 0.0,
    }

    if q_type == "definition":
        prefs["product"] += 1.4
        prefs["home"] += 0.2

    elif q_type == "service_listing":
        prefs["service"] += 1.4
        prefs["home"] += 0.3

    elif q_type == "product_listing":
        prefs["product"] += 1.4
        prefs["home"] += 0.2

    elif q_type == "about_company":
        prefs["about"] += 1.5
        prefs["home"] += 0.2

    elif q_type == "case_study":
        prefs["case_study"] += 1.5

    elif q_type == "blog":
        prefs["blog"] += 1.5

    else:
        if any(x in q_low for x in ["service", "services", "solution", "solutions", "consultation", "audit", "erp", "marketing"]):
            prefs["service"] += 1.0
        if any(x in q_low for x in ["product", "products", "2loyal", "artrixx", "platform", "software"]):
            prefs["product"] += 1.0
        if any(x in q_low for x in ["about", "mission", "vision", "company", "background"]):
            prefs["about"] += 1.0
        if any(x in q_low for x in ["blog", "blogs", "article", "articles", "insight", "news"]):
            prefs["blog"] += 1.0
        if any(x in q_low for x in ["case study", "case studies", "portfolio", "project", "projects", "success story"]):
            prefs["case_study"] += 1.0

    return prefs

def detect_allowed_page_types(query: str):
    q_type = classify_query_type(query)
    entities = extract_query_entities(query)

    if q_type == "definition":
        if any(e in {"2loyal", "artrixx"} for e in entities):
            return {"product"}
        return {"product", "about", "home"}

    if q_type == "service_listing":
        return {"service", "home"}

    if q_type == "product_listing":
        return {"product", "home"}

    if q_type == "about_company":
        return {"about", "home"}

    if q_type == "case_study":
        return {"case_study"}

    if q_type == "blog":
        return {"blog"}

    return None

## 13) Improved hybrid retrieval

In [ ]:
def improved_hybrid_retrieve(query: str, final_k: int = 8, dense_k: int = 16, bm25_k: int = 16):
    expanded_queries = expand_query(query)
    page_prefs = detect_page_type_preferences(query)
    allowed_page_types = detect_allowed_page_types(query)
    query_type = classify_query_type(query)
    query_entities = extract_query_entities(query)

    candidate_map = {}

    # Dense retrieval with MMR
    for q in expanded_queries:
        dense_docs = vectorstore.max_marginal_relevance_search(
            q,
            k=dense_k,
            fetch_k=max(dense_k * 3, 24),
            lambda_mult=0.65
        )

        for rank, doc in enumerate(dense_docs, start=1):
            page_type = doc.metadata.get("page_type", "general")
            if allowed_page_types is not None and page_type not in allowed_page_types:
                continue

            cid = doc.metadata["chunk_id"]
            if cid not in candidate_map:
                candidate_map[cid] = {
                    "doc": doc,
                    "dense_ranks": [],
                    "bm25_ranks": [],
                }
            candidate_map[cid]["dense_ranks"].append(rank)

    # BM25 retrieval
    expanded_query_text = " ".join(expanded_queries)
    bm25_scores = bm25.get_scores(simple_tokenize(expanded_query_text))
    bm25_top_idx = np.argsort(bm25_scores)[::-1][:bm25_k * 3]

    kept_bm25 = 0
    for rank, idx in enumerate(bm25_top_idx, start=1):
        doc = chunk_docs[idx]
        page_type = doc.metadata.get("page_type", "general")
        if allowed_page_types is not None and page_type not in allowed_page_types:
            continue

        cid = doc.metadata["chunk_id"]
        if cid not in candidate_map:
            candidate_map[cid] = {
                "doc": doc,
                "dense_ranks": [],
                "bm25_ranks": [],
            }
        candidate_map[cid]["bm25_ranks"].append(rank)
        kept_bm25 += 1
        if kept_bm25 >= bm25_k:
            break

    rows = []
    for cid, item in candidate_map.items():
        doc = item["doc"]

        dense_score = 0.0
        if item["dense_ranks"]:
            dense_score = sum(1.0 / (40 + r) for r in item["dense_ranks"])

        bm25_score = 0.0
        if item["bm25_ranks"]:
            bm25_score = sum(1.0 / (40 + r) for r in item["bm25_ranks"])

        page_type = doc.metadata.get("page_type", "general")
        page_boost = page_prefs.get(page_type, 0.0)

        section_heading = doc.metadata.get("section_heading", "").lower()
        title_text = doc.metadata.get("title", "").lower()
        source_url = doc.metadata.get("source", "").lower()
        doc_text_low = doc.page_content.lower()

        heading_boost = 0.0
        title_boost = 0.0
        url_boost = 0.0
        exact_term_boost = 0.0
        entity_boost = 0.0
        query_type_boost = 0.0

        for term in expanded_queries:
            t = term.lower().strip()
            if len(t) < 3:
                continue

            if t in section_heading:
                heading_boost += 0.18
            if t in title_text:
                title_boost += 0.14
            if t in source_url:
                url_boost += 0.12

            count_in_doc = doc_text_low.count(t)
            exact_term_boost += min(count_in_doc * 0.025, 0.15)

        # stronger exact entity handling
        for ent in query_entities:
            if ent in title_text:
                entity_boost += 0.35
            if ent in section_heading:
                entity_boost += 0.40
            if ent in source_url:
                entity_boost += 0.30

            ent_count = doc_text_low.count(ent)
            entity_boost += min(ent_count * 0.06, 0.30)

        # query-type-specific boosts
        if query_type == "definition":
            if any(x in section_heading for x in ["what is", "overview", "about", "introduction"]):
                query_type_boost += 0.20
            if page_type == "product":
                query_type_boost += 0.18

        elif query_type == "service_listing":
            if any(x in section_heading for x in ["services", "solutions", "what we offer", "our services"]):
                query_type_boost += 0.25
            if page_type == "service":
                query_type_boost += 0.18

        elif query_type == "product_listing":
            if any(x in section_heading for x in ["products", "our products", "platforms"]):
                query_type_boost += 0.25
            if page_type == "product":
                query_type_boost += 0.18

        elif query_type == "about_company":
            if any(x in section_heading for x in ["about us", "mission", "vision", "who we are"]):
                query_type_boost += 0.25
            if page_type == "about":
                query_type_boost += 0.18

        elif query_type == "case_study":
            if any(x in section_heading for x in ["case study", "case studies", "portfolio", "project"]):
                query_type_boost += 0.25
            if page_type == "case_study":
                query_type_boost += 0.18

        total_score = (
            0.40 * dense_score +
            0.22 * bm25_score +
            0.08 * page_boost +
            0.06 * heading_boost +
            0.05 * title_boost +
            0.03 * url_boost +
            0.03 * exact_term_boost +
            0.08 * entity_boost +
            0.05 * query_type_boost
        )

        rows.append({
            "chunk_id": cid,
            "doc": doc,
            "dense_score": dense_score,
            "bm25_score": bm25_score,
            "page_boost": page_boost,
            "heading_boost": heading_boost,
            "title_boost": title_boost,
            "url_boost": url_boost,
            "exact_term_boost": exact_term_boost,
            "entity_boost": entity_boost,
            "query_type_boost": query_type_boost,
            "total_score": total_score,
        })

    rows = sorted(rows, key=lambda x: x["total_score"], reverse=True)

    final = []
    seen_keys = set()

    for row in rows:
        key = (
            row["doc"].metadata.get("source"),
            row["doc"].metadata.get("section_heading"),
            row["doc"].metadata.get("local_chunk_index"),
        )
        if key in seen_keys:
            continue
        seen_keys.add(key)
        final.append(row)
        if len(final) >= final_k:
            break

    return final

## 14) Retrieval debug test

In [ ]:
test_query = "What services does CrocoIT offer?"
results = improved_hybrid_retrieve(test_query, final_k=10)

for i, item in enumerate(results, start=1):
    doc = item["doc"]
    print(f"\n--- Candidate {i} ---")
    print("Title:", doc.metadata["title"])
    print("Page type:", doc.metadata["page_type"])
    print("Section:", doc.metadata["section_heading"])
    print("Source:", doc.metadata["source"])
    print(
        "Scores:",
        "dense=", round(item["dense_score"], 4),
        "bm25=", round(item["bm25_score"], 4),
        "page=", round(item["page_boost"], 4),
        "heading=", round(item["heading_boost"], 4),
        "exact=", round(item["exact_term_boost"], 4),
        "total=", round(item["total_score"], 4)
    )
    print(doc.page_content[:800], "...")


--- Candidate 1 ---
Title: Solutions – CrocoIT
Page type: service
Section: Solutions – CrocoIT
Source: https://crocoit.com/solutions
Scores: dense= 0.2457 bm25= 0.0244 page= 1.4 heading= 0.36 exact= 0.275 total= 0.3866
Title: Solutions – CrocoIT
URL: https://crocoit.com/solutions
Page Type: service
Section Heading: Solutions – CrocoIT

ions across digital channels
Maximize the potential of your online marketplace with our marketplace management expertise. We offer end-to-end services to optimize vendor performance, manage listings, and implement strategic growth initiatives. Stay ahead in the competitive e-commerce landscape with our dedicated management solutions.
Streamline your operations with our state-of-the-art ERP solutio ...

--- Candidate 2 ---
Title: Solutions – CrocoIT
Page type: service
Section: Solutions – CrocoIT
Source: https://crocoit.com/solutions
Scores: dense= 0.2416 bm25= 0.0238 page= 1.4 heading= 0.36 exact= 0.275 total= 0.3848
Title: Solutions – CrocoIT
URL: http

## 15) Load the LLM

In [ ]:
if not torch.cuda.is_available():
    raise RuntimeError("Please switch Colab to GPU runtime before running this cell.")

# Load generator
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.float16
)

tokenizer = AutoTokenizer.from_pretrained(LLM_MODEL_NAME)

model = AutoModelForCausalLM.from_pretrained(
    LLM_MODEL_NAME,
    quantization_config=bnb_config,
    device_map="auto"
)

generator = pipeline(
    "text-generation",
    model=model,
    tokenizer=tokenizer,
    max_new_tokens=260,
    do_sample=False,
    temperature=0.1,
    repetition_penalty=1.08,
    pad_token_id=tokenizer.eos_token_id
)

# Load reranker
reranker = CrossEncoder(
    RERANKER_MODEL_NAME,
    max_length=512,
    device="cuda" if torch.cuda.is_available() else "cpu"
)

print("LLM loaded successfully.")
print("Reranker loaded successfully.")

Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/393 [00:00<?, ?it/s]

LLM loaded successfully.
Reranker loaded successfully.


## 16) Prompting helpers

In [ ]:
def detect_language(query: str) -> str:
    return "en"

def build_context(retrieved_items):
    context_parts = []
    for i, item in enumerate(retrieved_items, start=1):
        doc = item["doc"]
        context_parts.append(
            f"[Source {i}]\n"
            f"Title: {doc.metadata.get('title', '')}\n"
            f"URL: {doc.metadata.get('source', '')}\n"
            f"Page Type: {doc.metadata.get('page_type', '')}\n"
            f"Section Heading: {doc.metadata.get('section_heading', '')}\n"
            f"Content:\n{doc.page_content}\n"
        )
    return "\n\n".join(context_parts)

def build_messages(query: str, context: str):
    system_prompt = (
        "You are a RAG assistant that must answer only in English and only from the provided context.\n"
        "Every claim must be supported by the retrieved content.\n"
        "If the answer is not clearly supported, reply exactly with: I do not have enough information from the available content.\n"
        "Do not hallucinate.\n"
        "Be concise, accurate, and structured."
    )
    user_prompt = (
        f"Question:\n{query}\n\n"
        f"Context:\n{context}\n\n"
        "Answer in English using only the context."
    )

    return [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": user_prompt},
    ]

def rerank_candidates(query: str, retrieved_items, top_n: int = 3):
    if len(retrieved_items) == 0:
        return []

    pairs = [(query, item["doc"].page_content) for item in retrieved_items]
    rerank_scores = reranker.predict(pairs, batch_size=8, show_progress_bar=False)

    reranked = []
    for item, rr_score in zip(retrieved_items, rerank_scores):
        updated = dict(item)
        updated["rerank_score"] = float(rr_score)
        reranked.append(updated)

    reranked = sorted(
        reranked,
        key=lambda x: (x["rerank_score"], x["total_score"]),
        reverse=True
    )
    return reranked[:top_n]

def should_abstain(query: str, retrieved_items):
    if len(retrieved_items) == 0:
        return True

    top = retrieved_items[0]
    top_rr = float(top.get("rerank_score", 0.0))
    top_ret = float(top.get("total_score", 0.0))
    page_type = top["doc"].metadata.get("page_type", "general")
    query_type = classify_query_type(query)
    entities = extract_query_entities(query)

    second_rr = float(retrieved_items[1].get("rerank_score", 0.0)) if len(retrieved_items) > 1 else -999.0
    rr_gap = top_rr - second_rr

    doc_text_low = top["doc"].page_content.lower()
    entity_match = any(ent in doc_text_low for ent in entities) if entities else False

    # abstention rules
    if top_rr < -2.0:
        return True

    if top_ret < 0.03:
        return True

    if query_type in {"definition", "product_listing"} and entities and not entity_match:
        return True

    if query_type == "definition" and page_type not in {"product", "about", "home"}:
        return True

    if query_type == "service_listing" and page_type not in {"service", "home"}:
        return True

    if rr_gap < 0.02 and top_rr < 0.2:
        return True

    return False

## 17) Answer function

In [ ]:
def answer_question(query: str, final_k: int = 3, candidate_k: int = 10):
    initial_retrieved = improved_hybrid_retrieve(query, final_k=candidate_k)
    retrieved = rerank_candidates(query, initial_retrieved, top_n=final_k)

    if should_abstain(query, retrieved):
        return "I do not have enough information from the available content.", []

    context = build_context(retrieved)
    messages = build_messages(query, context)

    prompt = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

    output = generator(prompt)[0]["generated_text"]
    answer = output[len(prompt):].strip()

    sources = []
    for item in retrieved:
        doc = item["doc"]
        sources.append({
            "title": doc.metadata.get("title"),
            "url": doc.metadata.get("source"),
            "page_type": doc.metadata.get("page_type"),
            "section_heading": doc.metadata.get("section_heading"),
            "retrieval_score": item.get("total_score"),
            "rerank_score": item.get("rerank_score"),
        })

    return answer, sources

## 18) Demo questions

In [ ]:
queries = [
    "What services does CrocoIT offer?",
    "What products does CrocoIT mention?",
    "What does the About Us page say about CrocoIT?",
    "What is written in the case studies section?",
    "What is 2Loyal?",
]

for q in queries:
    print("=" * 100)
    print("QUESTION:", q)
    answer, sources = answer_question(q, final_k=3, candidate_k=10)

    print("\nANSWER:\n", answer)
    print("\nSOURCES:")
    if len(sources) == 0:
        print("- No sources returned due to abstention")
    else:
        for s in sources:
            print(
                "-",
                s["title"], "|",
                s["page_type"], "|",
                s["section_heading"], "|",
                "retrieval=", round(float(s["retrieval_score"]), 4), "|",
                "rerank=", round(float(s["rerank_score"]), 4)
            )
    print()

QUESTION: What services does CrocoIT offer?


Both `max_new_tokens` (=260) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



ANSWER:
 CrocoIT offers expert consultation and audit services, including digital transformation consultation, IT audits, and business process evaluations. They also provide marketplace management expertise for optimizing vendor performance, managing listings, and implementing strategic growth initiatives. Additionally, they offer state-of-the-art ERP solutions to integrate and automate core business processes.

SOURCES:
- Solutions – CrocoIT | service | Solutions – CrocoIT | retrieval= 0.3833 | rerank= 0.9926
- Solutions – CrocoIT | service | Solutions – CrocoIT | retrieval= 0.3866 | rerank= 0.9925
- Solutions – CrocoIT | service | Solutions – CrocoIT | retrieval= 0.3833 | rerank= 0.98

QUESTION: What products does CrocoIT mention?


Both `max_new_tokens` (=260) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



ANSWER:
 CrocoIT mentions AI Virtual Makeup, AI Skin Analysis, AI Assistant & Chatbot, and a Magento Mobile App Builder as products.

SOURCES:
- Our Products – CrocoIT | product | Our Products – CrocoIT | retrieval= 0.3804 | rerank= 0.9458
- Our Products – CrocoIT | product | Our Products – CrocoIT | retrieval= 0.3848 | rerank= 0.8804
- Our Products – CrocoIT | product | Our Products – CrocoIT | retrieval= 0.3864 | rerank= 0.8428

QUESTION: What does the About Us page say about CrocoIT?


Both `max_new_tokens` (=260) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



ANSWER:
 The About Us page states that CrocoIT's team of seasoned consultants provides in-depth digital transformation consultation, IT audits, and business process evaluations to ensure your technology strategy drives growth. It also mentions benefiting from a data-driven approach and actionable insights tailored to your industry.

SOURCES:
- About Us – CrocoIT | about | About Us – CrocoIT | retrieval= 0.3525 | rerank= 0.9922
- CrocoIT - CrocoIT | home | CrocoIT - CrocoIT | retrieval= 0.2156 | rerank= 0.4277
- CrocoIT - CrocoIT | home | CrocoIT - CrocoIT | retrieval= 0.2188 | rerank= 0.326

QUESTION: What is written in the case studies section?

ANSWER:
 I do not have enough information from the available content.

SOURCES:
- No sources returned due to abstention

QUESTION: What is 2Loyal?

ANSWER:
 I do not have enough information from the available content.

SOURCES:
- No sources returned due to abstention



## 19) Interactive chatbot

In [ ]:
while True:
    user_q = input("Ask a question (or type 'exit'): ").strip()
    if user_q.lower() == "exit":
        break

    answer, sources = answer_question(user_q, final_k=3, candidate_k=10)

    print("\nANSWER:\n")
    print(answer)

    print("\nSOURCES:")
    if len(sources) == 0:
        print("- No sources returned due to abstention")
    else:
        for s in sources:
            print(
                "-",
                s["title"], "|",
                s["page_type"], "|",
                s["section_heading"], "|",
                "retrieval=", round(float(s["retrieval_score"]), 4), "|",
                "rerank=", round(float(s["rerank_score"]), 4), "|",
                s["url"]
            )

    print("\n" + "=" * 100 + "\n")

Ask a question (or type 'exit'): exit


## 20) Retrieval inspection table

In [ ]:
def inspect_retrieval(query: str, candidate_k: int = 10, final_k: int = 3):
    initial = improved_hybrid_retrieve(query, final_k=candidate_k)
    reranked = rerank_candidates(query, initial, top_n=final_k)

    out = []
    for item in reranked:
        doc = item["doc"]
        out.append({
            "title": doc.metadata.get("title"),
            "page_type": doc.metadata.get("page_type"),
            "section_heading": doc.metadata.get("section_heading"),
            "source": doc.metadata.get("source"),
            "dense_score": round(item["dense_score"], 4),
            "bm25_score": round(item["bm25_score"], 4),
            "page_boost": round(item["page_boost"], 4),
            "heading_boost": round(item["heading_boost"], 4),
            "title_boost": round(item["title_boost"], 4),
            "url_boost": round(item["url_boost"], 4),
            "exact_term_boost": round(item["exact_term_boost"], 4),
            "entity_boost": round(item["entity_boost"], 4),
            "query_type_boost": round(item["query_type_boost"], 4),
            "retrieval_score": round(item["total_score"], 4),
            "rerank_score": round(item["rerank_score"], 4),
        })

    return pd.DataFrame(out)

inspect_retrieval("What is 2Loyal?")

,title,page_type,section_heading,source,dense_score,bm25_score,page_boost,heading_boost,title_boost,url_boost,exact_term_boost,entity_boost,query_type_boost,retrieval_score,rerank_score
0,Our Products – CrocoIT,product,Our Products – CrocoIT,https://crocoit.com/our-products,0.2449,0.0233,1.4,0.36,0.28,0.24,0.25,0.0,0.18,0.2744,0.0001
1,Our Products – CrocoIT,product,Our Products – CrocoIT,https://crocoit.com/our-products,0.2585,0.0244,1.4,0.36,0.28,0.24,0.25,0.0,0.18,0.2801,0.0000
2,Our Products – CrocoIT,product,Our Products – CrocoIT,https://crocoit.com/our-products,0.2672,0.0222,1.4,0.36,0.28,0.24,0.20,0.0,0.18,0.2816,0.0000


## 21) Context retrieval helper for evaluation

In [ ]:
def get_retrieved_contexts(query: str, final_k: int = 3, candidate_k: int = 10):
    initial_retrieved = improved_hybrid_retrieve(query, final_k=candidate_k)
    retrieved = rerank_candidates(query, initial_retrieved, top_n=final_k)

    if should_abstain(query, retrieved):
        return [], []

    contexts = [item["doc"].page_content for item in retrieved]
    sources = [
        {
            "title": item["doc"].metadata.get("title", ""),
            "url": item["doc"].metadata.get("source", ""),
            "page_type": item["doc"].metadata.get("page_type", ""),
            "section_heading": item["doc"].metadata.get("section_heading", ""),
            "retrieval_score": item["total_score"],
            "rerank_score": item["rerank_score"],
        }
        for item in retrieved
    ]
    return contexts, sources

## 22) New evaluation dataset

In [ ]:
advanced_eval_samples = [
    {
        "question": "What services does CrocoIT offer?",
        "ground_truth": "CrocoIT offers business and technology solutions such as consultation and audit, marketing planning, marketplace management, and ERP-related solutions.",
        "type": "standard"
    },
    {
        "question": "What products does CrocoIT mention?",
        "ground_truth": "CrocoIT mentions products such as 2Loyal and ARTRIXX if they appear in the retrieved website content.",
        "type": "standard"
    },
    {
        "question": "What does the About Us page say about CrocoIT?",
        "ground_truth": "The answer should summarize CrocoIT's company identity, mission, goals, or value proposition based on the About Us content.",
        "type": "standard"
    },
    {
        "question": "Does CrocoIT mention offering cloud hosting services?",
        "ground_truth": "If this is not clearly stated in the retrieved content, the system should say it does not have enough information.",
        "type": "negative_rejection"
    },
    {
        "question": "What is CrocoIT's office address in Germany?",
        "ground_truth": "The system should say it does not have enough information from the available content unless that is explicitly found in the website content.",
        "type": "negative_rejection"
    },
    {
        "question": "Combine what the Services page and About Us page say about CrocoIT's business value.",
        "ground_truth": "The answer should integrate information from more than one retrieved source and combine service offerings with company mission or value proposition.",
        "type": "information_integration"
    },
    {
        "question": "Summarize CrocoIT's services and products together.",
        "ground_truth": "The answer should combine service-related and product-related information from multiple retrieved chunks.",
        "type": "information_integration"
    },
]

## 23) Generate evaluation predictions

In [ ]:
eval_rows = []

for sample in advanced_eval_samples:
    q = sample["question"]
    answer, sources = answer_question(q, final_k=3, candidate_k=10)
    contexts, retrieved_sources = get_retrieved_contexts(q, final_k=3, candidate_k=10)

    eval_rows.append({
        "question": q,
        "answer": answer,
        "contexts": contexts,
        "ground_truth": sample["ground_truth"],
        "type": sample["type"],
        "sources": retrieved_sources
    })

eval_df = pd.DataFrame(eval_rows)
eval_df.head()

Both `max_new_tokens` (=260) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=260) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=260) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=260) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both

,question,answer,contexts,ground_truth,type,sources
0,What services does CrocoIT offer?,CrocoIT offers expert consultation and audit s...,[Title: Solutions – CrocoIT\nURL: https://croc...,CrocoIT offers business and technology solutio...,standard,"[{'title': 'Solutions – CrocoIT', 'url': 'http..."
1,What products does CrocoIT mention?,"CrocoIT mentions AI Virtual Makeup, AI Skin An...",[Title: Our Products – CrocoIT\nURL: https://c...,CrocoIT mentions products such as 2Loyal and A...,standard,"[{'title': 'Our Products – CrocoIT', 'url': 'h..."
2,What does the About Us page say about CrocoIT?,The About Us page states that CrocoIT's team o...,[Title: About Us – CrocoIT\nURL: https://croco...,The answer should summarize CrocoIT's company ...,standard,"[{'title': 'About Us – CrocoIT', 'url': 'https..."
3,Does CrocoIT mention offering cloud hosting se...,"No, CrocoIT does not explicitly mention offeri...",[Title: Solutions – CrocoIT\nURL: https://croc...,If this is not clearly stated in the retrieved...,negative_rejection,"[{'title': 'Solutions – CrocoIT', 'url': 'http..."
4,What is CrocoIT's office address in Germany?,I do not have enough information from the avai...,[],The system should say it does not have enough ...,negative_rejection,[]


## 24) Save evaluation predictions

In [ ]:
eval_path = DATA_DIR / "advanced_eval_predictions.json"

with open(eval_path, "w", encoding="utf-8") as f:
    json.dump(eval_rows, f, ensure_ascii=False, indent=2)

print("Saved:", eval_path)

Saved: /content/crocoit_rag_final/data/advanced_eval_predictions.json


## 25) Text normalization utilities

In [ ]:
def normalize_text(text: str) -> str:
    text = text.lower().strip()
    text = re.sub(r"[^\w\u0600-\u06FF\s]", " ", text)
    text = re.sub(r"\s+", " ", text)
    return text

def jaccard_similarity(a: str, b: str):
    sa = set(normalize_text(a).split())
    sb = set(normalize_text(b).split())
    if not sa or not sb:
        return 0.0
    return len(sa & sb) / len(sa | sb)

def context_concat(contexts):
    return " ".join(contexts)

## 26) Answer relevance

In [ ]:
def answer_relevance_score(question: str, answer: str):
    return jaccard_similarity(question, answer)

eval_df["answer_relevance_score"] = eval_df.apply(
    lambda row: answer_relevance_score(row["question"], row["answer"]),
    axis=1
)

eval_df[["question", "type", "answer_relevance_score"]]

,question,type,answer_relevance_score
0,What services does CrocoIT offer?,standard,0.065217
1,What products does CrocoIT mention?,standard,0.100000
2,What does the About Us page say about CrocoIT?,standard,0.108696
3,Does CrocoIT mention offering cloud hosting se...,negative_rejection,0.466667
4,What is CrocoIT's office address in Germany?,negative_rejection,0.000000
5,Combine what the Services page and About Us pa...,information_integration,0.204082
6,Summarize CrocoIT's services and products toge...,information_integration,0.102564


## 27) Faithfulness / fidelity

In [ ]:
def faithfulness_score(answer: str, contexts: list):
    return jaccard_similarity(answer, context_concat(contexts))

eval_df["faithfulness_score"] = eval_df.apply(
    lambda row: faithfulness_score(row["answer"], row["contexts"]),
    axis=1
)

eval_df[["question", "type", "faithfulness_score"]]

,question,type,faithfulness_score
0,What services does CrocoIT offer?,standard,0.278689
1,What products does CrocoIT mention?,standard,0.131579
2,What does the About Us page say about CrocoIT?,standard,0.372549
3,Does CrocoIT mention offering cloud hosting se...,negative_rejection,0.024194
4,What is CrocoIT's office address in Germany?,negative_rejection,0.000000
5,Combine what the Services page and About Us pa...,information_integration,0.300000
6,Summarize CrocoIT's services and products toge...,information_integration,0.227642


## 28) Context relevance

In [ ]:
def context_relevance_score(question: str, contexts: list):
    joined = context_concat(contexts)
    return jaccard_similarity(question, joined)

eval_df["context_relevance_score"] = eval_df.apply(
    lambda row: context_relevance_score(row["question"], row["contexts"]),
    axis=1
)

eval_df[["question", "type", "context_relevance_score"]]

,question,type,context_relevance_score
0,What services does CrocoIT offer?,standard,0.026316
1,What products does CrocoIT mention?,standard,0.017391
2,What does the About Us page say about CrocoIT?,standard,0.050000
3,Does CrocoIT mention offering cloud hosting se...,negative_rejection,0.017094
4,What is CrocoIT's office address in Germany?,negative_rejection,0.000000
5,Combine what the Services page and About Us pa...,information_integration,0.078431
6,Summarize CrocoIT's services and products toge...,information_integration,0.042735


## 29) Negative rejection

In [ ]:
REJECTION_PATTERNS = [
    "i do not have enough information",
    "not enough information",
    "i don't have enough information",
]

def is_rejection(answer: str):
    ans = answer.lower()
    return any(p.lower() in ans for p in REJECTION_PATTERNS)

negative_df = eval_df[eval_df["type"] == "negative_rejection"].copy()
negative_df["correct_rejection"] = negative_df["answer"].apply(is_rejection)

negative_df[["question", "answer", "correct_rejection"]]

,question,answer,correct_rejection
3,Does CrocoIT mention offering cloud hosting se...,"No, CrocoIT does not explicitly mention offeri...",False
4,What is CrocoIT's office address in Germany?,I do not have enough information from the avai...,True


## 30) Noise robustness

In [ ]:
NOISE_TEXT = """
This is irrelevant noise about cooking recipes, football, random astronomy facts, and unrelated company data.
It should not affect the answer if the RAG system is robust.
"""

def answer_question_with_noisy_context(query: str, final_k: int = 3, candidate_k: int = 10):
    initial_retrieved = improved_hybrid_retrieve(query, final_k=candidate_k)
    retrieved = rerank_candidates(query, initial_retrieved, top_n=final_k)

    if should_abstain(query, retrieved):
        return "I do not have enough information from the available content."

    context = build_context(retrieved)
    noisy_context = context + "\n\n[NOISE]\n" + NOISE_TEXT

    messages = build_messages(query, noisy_context)
    prompt = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

    output = generator(prompt)[0]["generated_text"]
    answer = output[len(prompt):].strip()
    return answer

noise_results = []

for sample in advanced_eval_samples:
    q = sample["question"]
    clean_answer, _ = answer_question(q, final_k=3, candidate_k=10)
    noisy_answer = answer_question_with_noisy_context(q, final_k=3, candidate_k=10)

    similarity = jaccard_similarity(clean_answer, noisy_answer)

    noise_results.append({
        "question": q,
        "type": sample["type"],
        "clean_answer": clean_answer,
        "noisy_answer": noisy_answer,
        "noise_robustness_similarity": similarity
    })

noise_df = pd.DataFrame(noise_results)
noise_df[["question", "type", "noise_robustness_similarity"]]

Both `max_new_tokens` (=260) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=260) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=260) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=260) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both

,question,type,noise_robustness_similarity
0,What services does CrocoIT offer?,standard,0.606557
1,What products does CrocoIT mention?,standard,0.608696
2,What does the About Us page say about CrocoIT?,standard,1.000000
3,Does CrocoIT mention offering cloud hosting se...,negative_rejection,0.933333
4,What is CrocoIT's office address in Germany?,negative_rejection,1.000000
5,Combine what the Services page and About Us pa...,information_integration,0.437500
6,Summarize CrocoIT's services and products toge...,information_integration,0.289474


## 31) Counterfactual robustness

In [ ]:
COUNTERFACTUAL_TEXT = """
False statement: CrocoIT was founded in 1890.
False statement: CrocoIT is headquartered on Mars.
False statement: CrocoIT only sells medical devices.
These statements are intentionally false and should be ignored.
"""

def answer_question_with_counterfactual_context(query: str, final_k: int = 3, candidate_k: int = 10):
    initial_retrieved = improved_hybrid_retrieve(query, final_k=candidate_k)
    retrieved = rerank_candidates(query, initial_retrieved, top_n=final_k)

    if should_abstain(query, retrieved):
        return "I do not have enough information from the available content."

    context = build_context(retrieved)
    altered_context = context + "\n\n[COUNTERFACTUAL TEST]\n" + COUNTERFACTUAL_TEXT

    messages = build_messages(query, altered_context)
    prompt = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

    output = generator(prompt)[0]["generated_text"]
    answer = output[len(prompt):].strip()
    return answer

FALSE_MARKERS = [
    "1890",
    "mars",
    "medical devices",
    "on mars"
]

counterfactual_results = []

for sample in advanced_eval_samples:
    q = sample["question"]
    cf_answer = answer_question_with_counterfactual_context(q, final_k=3, candidate_k=10)

    used_false_info = any(marker in cf_answer.lower() for marker in FALSE_MARKERS)

    counterfactual_results.append({
        "question": q,
        "type": sample["type"],
        "counterfactual_answer": cf_answer,
        "used_false_info": used_false_info
    })

counterfactual_df = pd.DataFrame(counterfactual_results)
counterfactual_df[["question", "type", "used_false_info"]]

Both `max_new_tokens` (=260) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=260) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=260) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=260) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both

,question,type,used_false_info
0,What services does CrocoIT offer?,standard,False
1,What products does CrocoIT mention?,standard,False
2,What does the About Us page say about CrocoIT?,standard,False
3,Does CrocoIT mention offering cloud hosting se...,negative_rejection,False
4,What is CrocoIT's office address in Germany?,negative_rejection,False
5,Combine what the Services page and About Us pa...,information_integration,False
6,Summarize CrocoIT's services and products toge...,information_integration,False


## 32) Information integration

In [ ]:
integration_rows = []

integration_questions = [x for x in advanced_eval_samples if x["type"] == "information_integration"]

for sample in integration_questions:
    q = sample["question"]
    answer, sources = answer_question(q, final_k=3, candidate_k=10)
    unique_source_count = len(set([s["url"] for s in sources if s["url"]]))

    integration_rows.append({
        "question": q,
        "answer": answer,
        "unique_source_count": unique_source_count,
        "sources": sources
    })

integration_df = pd.DataFrame(integration_rows)
integration_df[["question", "unique_source_count"]]

Both `max_new_tokens` (=260) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=260) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


,question,unique_source_count
0,Combine what the Services page and About Us pa...,2
1,Summarize CrocoIT's services and products toge...,2


## 33) Evaluation summary

In [ ]:
summary = {
    "avg_answer_relevance": float(eval_df["answer_relevance_score"].mean()),
    "avg_faithfulness": float(eval_df["faithfulness_score"].mean()),
    "avg_context_relevance": float(eval_df["context_relevance_score"].mean()),
    "negative_rejection_accuracy": float(negative_df["correct_rejection"].mean()) if len(negative_df) > 0 else None,
    "avg_noise_robustness_similarity": float(noise_df["noise_robustness_similarity"].mean()),
    "counterfactual_error_rate": float(counterfactual_df["used_false_info"].mean()),
    "avg_information_integration_sources": float(integration_df["unique_source_count"].mean()) if len(integration_df) > 0 else None,
}

summary

{'avg_answer_relevance': 0.14960363505172733,
 'avg_faithfulness': 0.19066461662518414,
 'avg_context_relevance': 0.033138218028512814,
 'negative_rejection_accuracy': 0.5,
 'avg_noise_robustness_similarity': 0.6965085781095647,
 'counterfactual_error_rate': 0.0,
 'avg_information_integration_sources': 2.0}

## 34) Pretty evaluation display

In [ ]:
print("=== MAIN METRICS ===")
for k, v in summary.items():
    print(f"{k}: {v}")

print("\n=== PER-QUESTION CORE SCORES ===")
display(eval_df[[
    "question",
    "type",
    "answer_relevance_score",
    "faithfulness_score",
    "context_relevance_score"
]])

print("\n=== NEGATIVE REJECTION ===")
display(negative_df[[
    "question",
    "correct_rejection",
    "answer"
]])

print("\n=== NOISE ROBUSTNESS ===")
display(noise_df[[
    "question",
    "noise_robustness_similarity"
]])

print("\n=== COUNTERFACTUAL ROBUSTNESS ===")
display(counterfactual_df[[
    "question",
    "used_false_info"
]])

print("\n=== INFORMATION INTEGRATION ===")
display(integration_df[[
    "question",
    "unique_source_count"
]])

=== MAIN METRICS ===
avg_answer_relevance: 0.14960363505172733
avg_faithfulness: 0.19066461662518414
avg_context_relevance: 0.033138218028512814
negative_rejection_accuracy: 0.5
avg_noise_robustness_similarity: 0.6965085781095647
counterfactual_error_rate: 0.0
avg_information_integration_sources: 2.0

=== PER-QUESTION CORE SCORES ===


,question,type,answer_relevance_score,faithfulness_score,context_relevance_score
0,What services does CrocoIT offer?,standard,0.065217,0.278689,0.026316
1,What products does CrocoIT mention?,standard,0.100000,0.131579,0.017391
2,What does the About Us page say about CrocoIT?,standard,0.108696,0.372549,0.050000
3,Does CrocoIT mention offering cloud hosting se...,negative_rejection,0.466667,0.024194,0.017094
4,What is CrocoIT's office address in Germany?,negative_rejection,0.000000,0.000000,0.000000
5,Combine what the Services page and About Us pa...,information_integration,0.204082,0.300000,0.078431
6,Summarize CrocoIT's services and products toge...,information_integration,0.102564,0.227642,0.042735



=== NEGATIVE REJECTION ===


,question,correct_rejection,answer
3,Does CrocoIT mention offering cloud hosting se...,False,"No, CrocoIT does not explicitly mention offeri..."
4,What is CrocoIT's office address in Germany?,True,I do not have enough information from the avai...



=== NOISE ROBUSTNESS ===


,question,noise_robustness_similarity
0,What services does CrocoIT offer?,0.606557
1,What products does CrocoIT mention?,0.608696
2,What does the About Us page say about CrocoIT?,1.000000
3,Does CrocoIT mention offering cloud hosting se...,0.933333
4,What is CrocoIT's office address in Germany?,1.000000
5,Combine what the Services page and About Us pa...,0.437500
6,Summarize CrocoIT's services and products toge...,0.289474



=== COUNTERFACTUAL ROBUSTNESS ===


,question,used_false_info
0,What services does CrocoIT offer?,False
1,What products does CrocoIT mention?,False
2,What does the About Us page say about CrocoIT?,False
3,Does CrocoIT mention offering cloud hosting se...,False
4,What is CrocoIT's office address in Germany?,False
5,Combine what the Services page and About Us pa...,False
6,Summarize CrocoIT's services and products toge...,False



=== INFORMATION INTEGRATION ===


,question,unique_source_count
0,Combine what the Services page and About Us pa...,2
1,Summarize CrocoIT's services and products toge...,2


## 35) Save evaluation reports

In [ ]:
eval_df.to_csv(DATA_DIR / "eval_core_metrics.csv", index=False)
negative_df.to_csv(DATA_DIR / "eval_negative_rejection.csv", index=False)
noise_df.to_csv(DATA_DIR / "eval_noise_robustness.csv", index=False)
counterfactual_df.to_csv(DATA_DIR / "eval_counterfactual.csv", index=False)
integration_df.to_csv(DATA_DIR / "eval_information_integration.csv", index=False)

with open(DATA_DIR / "eval_summary.json", "w", encoding="utf-8") as f:
    json.dump(summary, f, ensure_ascii=False, indent=2)

print("All evaluation reports saved in:", DATA_DIR)

All evaluation reports saved in: /content/crocoit_rag_final/data


## 36) Save README

In [ ]:
readme_text = """
# CrocoIT RAG Chatbot — English Only Version with Entity-Aware Retrieval

## Overview
This project builds an English-only Retrieval-Augmented Generation (RAG) chatbot over the public CrocoIT website.

## Main improvements
- stronger website text cleaning
- smaller section-aware chunks
- hard page-type filtering
- hybrid retrieval
- English query expansion
- second-stage reranking
- stronger grounded prompting
- top-3 final context selection
- entity-aware retrieval boosting
- query-type-specific retrieval rules
- abstention threshold for weak retrieval

## Retrieval pipeline
1. Crawl CrocoIT pages
2. Clean and structure content
3. Split into smaller section-aware chunks
4. Embed chunks with BGE-M3
5. Store vectors in FAISS
6. Retrieve candidates with:
   - MMR dense retrieval
   - BM25 keyword retrieval
   - hard page filtering
   - weighted hybrid fusion
   - entity-aware boosts
   - query-type-specific boosts
7. Rerank top candidates
8. Abstain when retrieval confidence is weak
9. Send only the top 3 reranked chunks to the generator

## Evaluation included
- Answer relevance
- Faithfulness / fidelity
- Context relevance
- Negative rejection
- Noise robustness
- Counterfactual robustness
- Information integration
""".strip()

with open(BASE_DIR / "README.md", "w", encoding="utf-8") as f:
    f.write(readme_text)

print("README saved to:", BASE_DIR / "README.md")
print(readme_text)

README saved to: /content/crocoit_rag_final/README.md
# CrocoIT RAG Chatbot — English Only Version with Entity-Aware Retrieval

## Overview
This project builds an English-only Retrieval-Augmented Generation (RAG) chatbot over the public CrocoIT website.

## Main improvements
- stronger website text cleaning
- smaller section-aware chunks
- hard page-type filtering
- hybrid retrieval
- English query expansion
- second-stage reranking
- stronger grounded prompting
- top-3 final context selection
- entity-aware retrieval boosting
- query-type-specific retrieval rules
- abstention threshold for weak retrieval

## Retrieval pipeline
1. Crawl CrocoIT pages
2. Clean and structure content
3. Split into smaller section-aware chunks
4. Embed chunks with BGE-M3
5. Store vectors in FAISS
6. Retrieve candidates with:
   - MMR dense retrieval
   - BM25 keyword retrieval
   - hard page filtering
   - weighted hybrid fusion
   - entity-aware boosts
   - query-type-specific boosts
7. Rerank top candida